In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print("Количество обучающих изображений:", len(train_dataset))
print("Количество тестовых изображений:", len(test_dataset))

In [ ]:
class PlantCNN(nn.Module):

    def __init__(self):
        super(PlantCNN, self).__init__()

        self.network = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(32 * 16 * 16, 128),

            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)


model = PlantCNN()

print(model)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Устройство для обучения:", device)

In [ ]:
num_epochs = 5

train_losses = []

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)

    train_losses.append(epoch_loss)

    print(
        f"Эпоха [{epoch + 1}/{num_epochs}], "
        f"Loss: {epoch_loss:.4f}"
    )

print("\nОбучение завершено!")

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, num_epochs + 1),
    train_losses,
    marker='o'
)

plt.title("CNN Training Loss")
plt.xlabel("Эпоха")
plt.ylabel("Loss")

plt.xticks(range(1, num_epochs + 1))
plt.grid(True)

plt.show()

In [ ]:
torch.save(
    model.state_dict(),
    "plant_cnn.pth"
)

print("Модель успешно сохранена в plant_cnn.pth")

In [ ]:
model.eval()

correct = 0
total = 0
test_loss = 0.0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()


accuracy = 100 * correct / total

average_test_loss = test_loss / total

print(f"Accuracy: {accuracy:.2f}%")
print(f"Loss: {average_test_loss:.4f}")

In [ ]:
results = {
    "Метрика": ["Accuracy", "Loss"],
    "Значение": [
        f"{accuracy:.2f}%",
        f"{average_test_loss:.4f}"
    ]
}

import pandas as pd

results_df = pd.DataFrame(results)

display(results_df)

In [ ]:
classes = (
    'Самолёт',
    'Автомобиль',
    'Птица',
    'Кошка',
    'Олень',
    'Собака',
    'Лягушка',
    'Лошадь',
    'Корабль',
    'Грузовик'
)

print("Классы CIFAR-10:")

for i, name in enumerate(classes):
    print(f"{i}: {name}")